# 🎓 Student Attendance & Performance Prediction

## Project Overview

This project uses Machine Learning to:
1. **Predict Academic Risk** — Classify students as AT RISK or SAFE based on attendance and performance (Classification using Logistic Regression)
2. **Predict Final Grade (G3)** — Estimate a student's final grade using internal assessment scores and behavior features (Regression using Linear Regression)
3. **Compare Multiple Models** — Evaluate Logistic Regression, Random Forest, and SVM to find the best classifier

### Dataset
- **Source**: UCI Machine Learning Repository — Student Performance (Mathematics)
- **Records**: 395 students
- **Features**: 33 attributes including demographics, social, school, and grade information

### Key Features Used
| Feature | Description |
|---------|-------------|
| `absences` | Number of school absences (0–75) |
| `studytime` | Weekly study time (1–4 scale) |
| `failures` | Number of past class failures (0–3) |
| `G1` | First period grade (0–20) |
| `G2` | Second period grade (0–20) |
| `G3` | Final grade (0–20) — target for regression |

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report,
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('student-mat.csv', sep=';')
print(f"Dataset shape: {df.shape}")
print(f"Number of students: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")
df.head()

In [ ]:
# Check data types and missing values
print("\n--- Data Info ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum().sum(), "total missing values")
print("\n--- Statistical Summary ---")
df.describe()

## 3. Feature Engineering

We create an **Attendance** feature from the `absences` column:
- Formula: `Attendance = 100 - (absences × 2)`
- Each absence is treated as a 2% reduction in attendance
- Values are clipped to a minimum of 0%

In [ ]:
df['Attendance'] = 100 - (df['absences'] * 2)
df['Attendance'] = df['Attendance'].clip(lower=0)

print("Attendance feature created:")
df[['absences', 'Attendance']].describe()

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution of Final Grades (G3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grade distribution
axes[0].hist(df['G3'], bins=20, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].axvline(x=10, color='red', linestyle='--', label='Pass threshold (10)')
axes[0].set_xlabel('Final Grade (G3)')
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Distribution of Final Grades')
axes[0].legend()

# Attendance distribution
axes[1].hist(df['Attendance'], bins=20, edgecolor='black', color='seagreen', alpha=0.7)
axes[1].axvline(x=75, color='red', linestyle='--', label='75% threshold')
axes[1].set_xlabel('Attendance (%)')
axes[1].set_ylabel('Number of Students')
axes[1].set_title('Distribution of Attendance')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats
pass_count = (df['G3'] >= 10).sum()
fail_count = (df['G3'] < 10).sum()
print(f"\nStudents who PASSED (G3 >= 10): {pass_count} ({pass_count/len(df)*100:.1f}%)")
print(f"Students who FAILED (G3 < 10):  {fail_count} ({fail_count/len(df)*100:.1f}%)")

### 4.2 Attendance vs Final Grade (Scatter Plot)

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['Attendance'], df['G3'], 
                      c=df['G3'], cmap='RdYlGn', alpha=0.6, edgecolors='black', linewidth=0.5)
plt.colorbar(scatter, label='Grade')
plt.xlabel('Attendance (%)')
plt.ylabel('Final Grade (G3)')
plt.title('Attendance vs Final Grade')
plt.axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Pass line')
plt.axvline(x=75, color='blue', linestyle='--', alpha=0.5, label='75% attendance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4.3 Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
correlation_cols = ['Attendance', 'studytime', 'failures', 'G1', 'G2', 'G3', 'absences', 'age']
corr_matrix = df[correlation_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print("\nKey correlations with G3 (Final Grade):")
g3_corr = corr_matrix['G3'].drop('G3').sort_values(ascending=False)
for feat, val in g3_corr.items():
    print(f"  {feat:15s}: {val:+.4f}")

### 4.4 Study Time vs Performance (Box Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df['studytime'], y=df['G3'], ax=axes[0], palette='Blues')
axes[0].set_xlabel('Study Time (1=low, 4=high)')
axes[0].set_ylabel('Final Grade (G3)')
axes[0].set_title('Study Time vs Performance')

sns.boxplot(x=df['failures'], y=df['G3'], ax=axes[1], palette='Reds')
axes[1].set_xlabel('Past Failures (0-3)')
axes[1].set_ylabel('Final Grade (G3)')
axes[1].set_title('Past Failures vs Performance')

plt.tight_layout()
plt.show()

print("\nStudy Time correlation with G3:", df[['studytime', 'G3']].corr().iloc[0, 1].__round__(4))
print("Failures correlation with G3:  ", df[['failures', 'G3']].corr().iloc[0, 1].__round__(4))

### 4.5 Grade Progression (G1 → G2 → G3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['G1'], df['G3'], alpha=0.5, color='blue', label='G1 vs G3')
z1 = np.polyfit(df['G1'], df['G3'], 1)
p1 = np.poly1d(z1)
axes[0].plot(sorted(df['G1']), p1(sorted(df['G1'])), 'r--', linewidth=2)
axes[0].set_xlabel('First Period Grade (G1)')
axes[0].set_ylabel('Final Grade (G3)')
axes[0].set_title('G1 vs G3 (with trend line)')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df['G2'], df['G3'], alpha=0.5, color='green', label='G2 vs G3')
z2 = np.polyfit(df['G2'], df['G3'], 1)
p2 = np.poly1d(z2)
axes[1].plot(sorted(df['G2']), p2(sorted(df['G2'])), 'r--', linewidth=2)
axes[1].set_xlabel('Second Period Grade (G2)')
axes[1].set_ylabel('Final Grade (G3)')
axes[1].set_title('G2 vs G3 (with trend line)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"G1-G3 correlation: {df['G1'].corr(df['G3']):.4f}")
print(f"G2-G3 correlation: {df['G2'].corr(df['G3']):.4f}")

## 5. Data Preparation

### 5.1 Create Target Variable (At_Risk)

A student is classified as **At Risk** if:
- Attendance < 75% **OR**
- Final Grade (G3) < 10

In [ ]:
required_percentage = 75

df['At_Risk'] = (
    (df['Attendance'] < required_percentage) | 
    (df['G3'] < 10)
).astype(int)

print("Target Variable Distribution:")
print(df['At_Risk'].value_counts())
print(f"\nSAFE (0): {(df['At_Risk'] == 0).sum()} students ({(df['At_Risk'] == 0).mean()*100:.1f}%)")
print(f"AT RISK (1): {(df['At_Risk'] == 1).sum()} students ({(df['At_Risk'] == 1).mean()*100:.1f}%)")

# Visualize
plt.figure(figsize=(6, 4))
df['At_Risk'].value_counts().plot(kind='bar', color=['green', 'red'], edgecolor='black')
plt.xticks([0, 1], ['SAFE (0)', 'AT RISK (1)'], rotation=0)
plt.ylabel('Count')
plt.title('At Risk Classification Distribution')
plt.show()

### 5.2 Define Features and Split Data

In [ ]:
features = ['Attendance', 'studytime', 'failures', 'G1', 'G2']

X = df[features]
y = df['At_Risk']      # Target for classification
g3 = df['G3']           # Target for regression

print(f"Features: {features}")
print(f"Feature matrix shape: {X.shape}")
print(f"Classification target shape: {y.shape}")
print(f"Regression target shape: {g3.shape}")

In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler")
print(f"Mean (train, after scaling): {X_train_scaled.mean(axis=0).round(2)}")
print(f"Std  (train, after scaling): {X_train_scaled.std(axis=0).round(2)}")

## 6. Model 1: Logistic Regression (Classification)

**Goal**: Predict whether a student is AT RISK (1) or SAFE (0)

In [ ]:
# Train Logistic Regression
clf_lr = LogisticRegression(random_state=42, max_iter=1000)
clf_lr.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = clf_lr.predict(X_test_scaled)

# Accuracy
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {acc_lr*100:.2f}%")

### 6.1 Confusion Matrix

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['SAFE', 'AT RISK'], 
            yticklabels=['SAFE', 'AT RISK'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression — Confusion Matrix')
plt.show()

print(f"True Negatives (correctly predicted SAFE):    {cm_lr[0][0]}")
print(f"False Positives (SAFE predicted as AT RISK):  {cm_lr[0][1]}")
print(f"False Negatives (AT RISK predicted as SAFE):  {cm_lr[1][0]}")
print(f"True Positives (correctly predicted AT RISK): {cm_lr[1][1]}")

### 6.2 Classification Report (Precision, Recall, F1-Score)

In [ ]:
print("Logistic Regression — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_lr, target_names=['SAFE', 'AT RISK']))

## 7. Model 2: Linear Regression (Grade Prediction)

**Goal**: Predict the final grade (G3) as a continuous value (0–20)

In [ ]:
# Train Linear Regression
reg_model = LinearRegression()
reg_model.fit(X_train_scaled, g3.loc[y_train.index])

# Predict on test set
g3_test_actual = g3.loc[y_test.index]
g3_pred = reg_model.predict(X_test_scaled)

# Evaluation Metrics
r2 = r2_score(g3_test_actual, g3_pred)
mae = mean_absolute_error(g3_test_actual, g3_pred)
rmse = np.sqrt(mean_squared_error(g3_test_actual, g3_pred))

print("Linear Regression — Performance Metrics")
print("=" * 45)
print(f"R² Score:                  {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error:   {rmse:.4f}")

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(g3_test_actual, g3_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
plt.plot([0, 20], [0, 20], 'r--', label='Perfect prediction')
plt.xlabel('Actual Grade (G3)')
plt.ylabel('Predicted Grade (G3)')
plt.title(f'Linear Regression — Actual vs Predicted (R² = {r2:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Model Comparison (Classification)

We compare **3 classifiers** on the same data:
1. Logistic Regression
2. Random Forest Classifier
3. Support Vector Machine (SVM)

In [ ]:
# Random Forest
clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rf.fit(X_train_scaled, y_train)
y_pred_rf = clf_rf.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, y_pred_rf)

# SVM
clf_svm = SVC(kernel='rbf', random_state=42)
clf_svm.fit(X_train_scaled, y_train)
y_pred_svm = clf_svm.predict(X_test_scaled)
acc_svm = accuracy_score(y_test, y_pred_svm)

print("Model Comparison — Classification Accuracy")
print("=" * 50)

comparison_data = {
    'Model': ['Logistic Regression', 'Random Forest', 'SVM (RBF)'],
    'Accuracy': [f'{acc_lr*100:.2f}%', f'{acc_rf*100:.2f}%', f'{acc_svm*100:.2f}%'],
    'Accuracy (raw)': [acc_lr, acc_rf, acc_svm]
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df[['Model', 'Accuracy']].to_string(index=False))

In [ ]:
# Bar chart comparison
models = ['Logistic\nRegression', 'Random\nForest', 'SVM\n(RBF)']
accuracies = [acc_lr * 100, acc_rf * 100, acc_svm * 100]
colors = ['steelblue', 'seagreen', 'coral']

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color=colors, edgecolor='black', width=0.5)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, 
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.ylabel('Accuracy (%)')
plt.title('Model Comparison — Classification Accuracy')
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Detailed classification reports for all models
print("\n" + "=" * 60)
print("RANDOM FOREST — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_rf, target_names=['SAFE', 'AT RISK']))

print("\n" + "=" * 60)
print("SVM (RBF) — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_svm, target_names=['SAFE', 'AT RISK']))

In [ ]:
# Confusion matrices for all 3 models side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, y_pred, name, color in zip(
    axes, 
    [y_pred_lr, y_pred_rf, y_pred_svm],
    ['Logistic Regression', 'Random Forest', 'SVM (RBF)'],
    ['Blues', 'Greens', 'Oranges']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, ax=ax,
                xticklabels=['SAFE', 'AT RISK'], 
                yticklabels=['SAFE', 'AT RISK'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAccuracy: {acc*100:.2f}%')

plt.tight_layout()
plt.show()

## 9. Sample Prediction

Let's test our models with a sample student:
- Attendance: 80%
- Study Time: 2
- Failures: 1
- G1: 10
- G2: 12

In [ ]:
sample = pd.DataFrame([[80, 2, 1, 10, 12]], columns=features)
sample_scaled = scaler.transform(sample)

# Classification prediction
risk_lr = clf_lr.predict(sample_scaled)
risk_rf = clf_rf.predict(sample_scaled)
risk_svm = clf_svm.predict(sample_scaled)

# Regression prediction
g3_pred_sample = reg_model.predict(sample_scaled)

print("Sample Student: Attendance=80%, StudyTime=2, Failures=1, G1=10, G2=12")
print("=" * 65)
print(f"\n🔹 Logistic Regression → {'AT RISK' if risk_lr[0] == 1 else 'SAFE'}")
print(f"🔹 Random Forest       → {'AT RISK' if risk_rf[0] == 1 else 'SAFE'}")
print(f"🔹 SVM (RBF)           → {'AT RISK' if risk_svm[0] == 1 else 'SAFE'}")
print(f"\n📊 Predicted Final Grade (G3): {g3_pred_sample[0]:.2f} / 20")
print(f"   Result: {'✅ PASS' if g3_pred_sample[0] >= 10 else '❌ FAIL'}")

## 10. Conclusion

### Key Findings

1. **G1 and G2 are the strongest predictors** of final grade (G3), with correlation > 0.80
2. **Past failures** negatively correlate with performance
3. **Attendance** (derived from absences) has moderate impact on grades
4. **Study time** has a weak but positive correlation with final grades

### Model Performance

| Model | Task | Key Metric |
|-------|------|------------|
| Logistic Regression | Risk Classification | ~87% Accuracy |
| Random Forest | Risk Classification | Compared above |
| SVM (RBF) | Risk Classification | Compared above |
| Linear Regression | Grade Prediction | R² shown above |

### Future Improvements
- Use more features (family support, internet access, etc.)
- Try ensemble methods (XGBoost, Gradient Boosting)
- Implement cross-validation for more robust evaluation
- Deploy the model as a web application (see `app.py`)